In [3]:
# -------------------- imports --------------------
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from models import CondSM
from bayes_opt import BayesianOptimization

# -------------------- helpers --------------------
def mse_loss_normalized(x, y):
    x = x.float()
    y = y.float()
    x_norm = (x - x.mean()) / (x.std(unbiased=False) + 1e-8)
    y_norm = (y - y.mean()) / (y.std(unbiased=False) + 1e-8)
    return torch.mean((y_norm - x_norm) ** 2).item()

# -------------------- load model --------------------
data_type = 'uni_1e7'
state_dict = torch.load(
    f'/gpfs/bwfor/work/ws/hd_gy283-my_data_recover/final_datasets/{data_type}/sm_{data_type}_baseline_0_epochs=1000_dp=0.1_bn=True_hd=90_ls=5_noise=True.pth',
    weights_only=False,
    map_location='cpu'
)

model = CondSM(state_dict['layer_dims'], state_dict['dropout'], state_dict['batch_norm'])
model.load_state_dict(state_dict['model_state_dict'])
model.eval()

# -------------------- range setup --------------------
v_min, v_max = -1.5, 1.5
N = 500
input_range = torch.linspace(v_min, v_max, N).unsqueeze(1)

input_idx = 0

# number of control electrodes
d = model.layer_dims[0] - 1

# -------------------- target --------------------
func_type = 'Sine'

if func_type == 'ReLU':
    def target_function(x):
        return nn.ReLU()(x)
elif func_type == 'Sigmoid':
    def target_function(x):
        return torch.sigmoid(4 * x)
elif func_type == 'Parabola':
    def target_function(x):
        return x ** 2
elif func_type == 'Sine':
    def target_function(x):
        return torch.sin(3 * x)
else:
    raise ValueError("Unknown func_type")

target = target_function(input_range)  # [N,1]

# -------------------- BASIS-OF-MODEL SETUP --------------------
# K basis elements; BayesOpt will pick K different control vectors
K = 4  # keep small; BO dims grow as K*d

# bounds for controls and mixing weights
pbounds = {}
for k in range(K):
    for i in range(d):
        pbounds[f'x{k}_{i}'] = (v_min, v_max)

w_bound = 2.0
for k in range(K):
    pbounds[f'a{k}'] = (-w_bound, w_bound)
pbounds['b'] = (-w_bound, w_bound)

# regularization strength (on controls and weights)
lam = 0.0
w_reg_scale = 0.1  # weight-reg is typically weaker than control-reg

# choose a simple composition g(·)
def g(u):
    # return u
    return torch.relu(u)
    # return torch.sigmoid(u)

# -------------------- objective for BayesOpt --------------------
def f(**kwargs):
    Ys = []
    reg_controls = 0.0

    # forward K times (one per basis element)
    for k in range(K):
        cv_list = [kwargs[f'x{k}_{i}'] for i in range(d)]
        reg_controls += float(np.sum(np.abs(cv_list)))  # L1 on controls

        cv_t = torch.tensor(cv_list, dtype=torch.float32).expand(N, -1)
        cv_t_l = cv_t[:, :input_idx]
        cv_t_r = cv_t[:, input_idx:]
        input_tensor = torch.cat([cv_t_l, input_range, cv_t_r], dim=1)

        model.eval()
        with torch.no_grad():
            y_out = model(input_tensor).detach()

        # take channel 0 as the curve and reshape to [N,1]
        Ys.append(y_out[..., 0].reshape(N, 1))

    # stack curves -> [N, K]
    Y = torch.cat(Ys, dim=1)

    # mixing weights + bias
    a = torch.tensor([kwargs[f'a{k}'] for k in range(K)], dtype=torch.float32).reshape(K, 1)  # [K,1]
    b = torch.tensor(float(kwargs['b']), dtype=torch.float32)

    # mix + compose -> [N,1]
    y_hat = (Y @ a).reshape(N, 1) + b
    y_hat = g(y_hat)

    # optional L1 on weights
    reg_w = float(np.sum(np.abs([kwargs[f'a{k}'] for k in range(K)])))

    loss = mse_loss_normalized(y_hat, target) + lam * (reg_controls + w_reg_scale * reg_w)
    return -loss  # BayesOpt maximizes

In [ ]:
# -------------------- run BayesOpt (your loop) --------------------
num_iters = 100
num_init_points = 10
num_runs = 20

opt_param_vals = []
opt_targets = []

for run in range(num_runs):
    optimizer = BayesianOptimization(
        f=f,
        pbounds=pbounds,
        verbose=1,
        random_state=np.random.randint(0, 2**31 - 1),
    )
    optimizer.maximize(
        init_points=num_init_points,
        n_iter=num_iters,
    )

    print('run done:', run)

    best_params = optimizer.max['params']
    opt_param_vals.append(np.array(list(best_params.values())))
    opt_targets.append(float(optimizer.max['target']))

|   iter    |  target   |   x0_0    |   x0_1    |   x0_2    |   x0_3    |   x0_4    |   x0_5    |   x1_0    |   x1_1    |   x1_2    |   x1_3    |   x1_4    |   x1_5    |   x2_0    |   x2_1    |   x2_2    |   x2_3    |   x2_4    |   x2_5    |   x3_0    |   x3_1    |   x3_2    |   x3_3    |   x3_4    |   x3_5    |    a0     |    a1     |    a2     |    a3     |     b     |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 5         | -1.945475 | 0.6515724 | 0.4630658 | 0.0280240 | 1.3780011 | -0.712151 | -1.388591 | -1.485560 | -0.582011 | -1.013173 | 1.4635323 | 0.4374440 | -1.018588 | -0.747895 | 0.8607811 | -0.152312 | 1.4564817 | 0.7122538 | -1.045378 | 0.6865523 

In [ ]:
# -------------------- visualize best run --------------------
best_run = int(np.argmax(opt_targets))
best_params = optimizer.max['params']  # note: this is last optimizer; better to rebuild from stored if needed

# If you want best of all runs, reconstruct params from stored list is messy;
# simplest: rerun one more optimizer with the best random_state if you stored it.
# For now, just show the LAST run's best:
def forward_from_params(best_params):
    Ys = []
    for k in range(K):
        cv_list = [best_params[f'x{k}_{i}'] for i in range(d)]
        cv_t = torch.tensor(cv_list, dtype=torch.float32).expand(N, -1)
        cv_t_l = cv_t[:, :input_idx]
        cv_t_r = cv_t[:, input_idx:]
        input_tensor = torch.cat([cv_t_l, input_range, cv_t_r], dim=1)
        with torch.no_grad():
            y_out = model(input_tensor).detach()
        Ys.append(y_out[..., 0].reshape(N, 1))
    Y = torch.cat(Ys, dim=1)
    a = torch.tensor([best_params[f'a{k}'] for k in range(K)], dtype=torch.float32).reshape(K, 1)
    b = torch.tensor(float(best_params['b']), dtype=torch.float32)
    y_hat = (Y @ a).reshape(N, 1) + b
    return g(y_hat)

with torch.no_grad():
    y_hat = forward_from_params(optimizer.max['params'])

plt.plot(input_range.numpy(), target.numpy(), label="target")
plt.plot(input_range.numpy(), y_hat.numpy(), label="basis-mix output")
plt.legend()
plt.show()